# Setup

In [0]:
%run ../config

In [0]:
%run ../helpers

# Imports

In [0]:
from pyspark.sql.functions import (
    col, lit, when, expr, count, count_distinct, max, min, sum, date_format, dayofweek, dayofmonth, month, quarter, format_string, initcap, try_element_at, current_timestamp, greatest, last_day, year
)

from pyspark.sql.types import StructType, StructField, StringType, TimestampType

# Variable

## Static Variable

In [0]:
SOURCE_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.crimes"
CONTROL_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.load_control"

SILVER_CA_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.community_area_boundaries"

In [0]:
# every Gold table that this notebook writes

# Fact Table

FACT_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.fact_crime"

# Dimension Tables

DIM_DATE_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.dim_date"
DIM_TIME_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.dim_time"
DIM_CRIME_TYPE_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.dim_crime_type"
DIM_LOCATION_DESCRIPTION_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.dim_location_description"
DIM_BLOCK_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.dim_block"
DIM_LOCATION_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.dim_location"
DIM_ARREST_STATUS_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.dim_arrest_status"

# Reference Table

REF_COMMUNITY_AREA_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.ref_community_area_boundaries"

In [0]:
# setup control schema for the control table in the gold layer

CONTROL_SCHEMA = StructType([
    StructField("table_name", StringType(), False),
    StructField("last_ingested_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False)
])

In [0]:
EXPECTED_SILVER_COLUMNS = [
    "id", "case_number", "arrest", "domestic", "updated_on", "_ingested_at",
    "incident_date", "incident_hour",
    "iucr", "primary_type", "description", "fbi_code",
    "beat", "district", "ward", "community_area",
    "location_description",
    "block", "block_hundred", "direction", "street_name", "street_type",
    "x_coordinate", "y_coordinate", "latitude", "longitude",
]

In [0]:
DATE_START = "2001-01-01"
DATE_END = "2040-12-31"

EXPECTED_COMMUNITY_AREAS = 77

# Widget

In [0]:
dbutils.widgets.dropdown(
    name="run_mode",
    defaultValue="incremental",
    choices=['incremental', 'full_reprocess'],
    label='run_mode'
)

In [0]:
run_mode = dbutils.widgets.get("run_mode")

# Watermark

In [0]:
def get_last_watermark():
    if not spark.catalog.tableExists(CONTROL_TABLE):
        print(f"[CONTROL TABLE] {CONTROL_TABLE} does not exist yet - first run, reading all of Silver")
        return None
    
    control_rows = spark.table(CONTROL_TABLE).filter(col("table_name") == "crimes")
    last_watermark = control_rows.agg(max("last_ingested_at")).collect()[0][0]

    if last_watermark is None:
        print(f"[CONTROL TABLE] no row for 'crimes' yet - first run, reading all of Silver")

    return last_watermark

In [0]:
if run_mode == "full_reprocess":
    last_watermark = None
    print("[FULL REPROCESS] ignoring the watermark - reading all of Silver")
else:
    last_watermark = get_last_watermark()
    print(f"[INCREMENTAL] reading Silver rows with _ingested_at > {last_watermark}")

# Data Validation

In [0]:
silver = spark.table(SOURCE_TABLE)

only check schema: a column of Silver that Gold needs and that is missing stops the run (FAIL): the Silver notebook changed and nothing after this point can be trusted.

Steps:
1. Set Up
2. Profile
3. Verify

In [0]:
# 1. [SET UP] the expected columns are EXPECTED_SILVER_COLUMNS

print(f"[INFO]-[T1][SET UP]: Silver must contain {len(EXPECTED_SILVER_COLUMNS)} columns that Gold reads")

In [0]:
# 2. [PROFILE] compare the real Silver columns with the expected ones

silver_types = dict(silver.dtypes)

missing_columns = [c for c in EXPECTED_SILVER_COLUMNS if c not in silver_types]

print(f"[INFO]-[T1][PROFILE]: missing columns: {missing_columns}")

In [0]:
# 3. [VERIFY] FAIL on a missing column

if missing_columns:
    raise ValueError(f"[ERROR]-[T1][VERIFY]: {SOURCE_TABLE} is missing columns: {missing_columns}")

print("[OK]-[T1][VERIFY]: Silver has every column Gold reads")

## Read the batch

In [0]:
batch_end = silver.agg(max("_ingested_at")).collect()[0][0]

if batch_end is None:
    dbutils.notebook.exit("[EXIT]: Silver table is EMPTY")

In [0]:
df_raw = silver.filter(col("_ingested_at") <= lit(batch_end))

if last_watermark is not None:
    df_raw = df_raw.filter(col("_ingested_at") > lit(last_watermark))

raw_rows = df_raw.count()
print(f"[INFO]: ({last_watermark}, {batch_end}] -> {raw_rows:,} Silver rows")

if raw_rows == 0:
    dbutils.notebook.exit("[EXIT]: no Silver rows after the watermark")

# Dimensional Model (Star Schema)

## Dimension Tables

### Static Dimension

`gold.dim_date`, `gold.dim_time` and `gold.ref_community_area_boundaries` never change with the crime data.

#### `dim_date`

Step: 
1. ~~Enrich~~ (every column is derived from the date itself)
2. Transform
3. Build
4. Load
5. Verify

In [0]:
# 2. [TRANSFORM]

days = spark.sql(f"SELECT explode(sequence(DATE'{DATE_START}', DATE'{DATE_END}', INTERVAL 1 DAY)) AS full_date")

day_in_week = dayofweek("full_date")

dim_date = days.select(
    date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
    col("full_date"),
    date_format("full_date", "d MMMM yyyy").alias("date_description"),
    dayofmonth("full_date").cast("tinyint").alias("day"),
    month("full_date").cast("tinyint").alias("month"),
    date_format("full_date", "MMMM").alias("month_name"),
    year("full_date").cast("smallint").alias("year"),
    date_format("full_date", "yyyy-MM").alias("month_year"),
    quarter("full_date").cast("tinyint").alias("quarter"),
    date_format("full_date", "EEEE").alias("weekday"),
    day_in_week.cast("tinyint").alias("day_in_week"),
    (day_in_week.isin(1, 7)).alias("is_weekend"), # Sunday = 1, Saturday = 7
    (day_in_week == 1).alias("is_first_day_of_week"), # Sunday
    (day_in_week == 7).alias("is_last_day_of_week"), # Saturday
    (dayofmonth("full_date") == 1).alias("is_first_day_of_month"),
    (col("full_date") == last_day("full_date")).alias("is_last_day_of_month"),
)

dim_date.printSchema()

In [0]:
# 3. [BUILD] check before write

expected_days = spark.sql(f"SELECT datediff(DATE'{DATE_END}', DATE'{DATE_START}') + 1").collect()[0][0]
date_rows, date_keys = dim_date.agg(count("*"), count_distinct("date_key")).collect()[0]

if date_rows != expected_days or date_keys != date_rows:
    raise ValueError(f"[ERROR]-[dim_date][BUILD]: expected {expected_days:,} rows with a unique date_key, found {date_rows:,} rows and {date_keys:,} distinct keys")

null_counts = dim_date.agg(*[sum(col(c).isNull().cast("int")).alias(c) for c in dim_date.columns]).collect()[0].asDict()
with_nulls = {c: n for c, n in null_counts.items() if n}

if with_nulls:
    raise ValueError(f"[ERROR]-[dim_date][BUILD]: NULL found in {with_nulls}")

print(f"[OK]-[dim_date][BUILD]: {date_rows:,} rows, unique date_key, no NULL")

In [0]:
# 4. [LOAD]

(
    dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_DATE_TABLE)
)

In [0]:
# 5. [VERIFY] read the table back

written = spark.table(DIM_DATE_TABLE).count()

if written != date_rows:
    raise ValueError(f"[ERROR]-[dim_date][VERIFY]: expected {date_rows:,} rows but {DIM_DATE_TABLE} has {written:,}")

print(f"[OK]-[dim_date][VERIFY]: {DIM_DATE_TABLE} has {written:,} rows")

#### `dim_time`

Steps:
1. ~~Enrich~~ (every column is derived from the hour itself)
2. Transform
3. Build
4. Load
5. Verify

In [0]:
# 2. [TRANSFORM]

hour = col("hour")

dim_time = spark.range(24).select(col("id").cast("int").alias("hour")).select(
    hour.cast("tinyint").alias("time_key"),
    hour.cast("tinyint").alias("hour_24"),
    (((hour + 11) % 12) + 1).cast("tinyint").alias("hour_12"),
    when(hour < 12, "AM").otherwise("PM").alias("am_pm"),
    format_string("%02d:00", hour).alias("hour_label"),
    (
        when(hour<=5, "Night (00-05)")
        .when(hour<=11, "Morning (06-11)")
        .when(hour<=17, "Afternoon (12-17)")
        .otherwise("Evening (18-23)")
    ).alias("day_part")
)

dim_time.printSchema()

In [0]:
# 3. [BUILD] the check before the write 

rows = {r["time_key"]: r for r in dim_time.collect()}

if sorted(rows) != list(range(24)):
    raise ValueError(f"[ERROR]-[dim_time][BUILD]: time_key must be 0...23, found {sorted(rows)}")

part_sizes = {r["day_part"]: r["count"] for r in dim_time.groupBy("day_part").count().collect()}
if set(part_sizes.values()) != {6} or len(part_sizes) != 4:
    raise ValueError(f"[ERROR]-[dim_time][BUILD]: every day_part must have 6 hours, found {part_sizes}")

print("[OK]-[dim_time][BUILD]: 24 rows, 12-hour clock and day parts are correct")

In [0]:
# 4. [LOAD]

(
    dim_time.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_TIME_TABLE)
)

In [0]:
# 5. [VERIFY] read the table back

written = spark.table(DIM_TIME_TABLE).count()

if written != 24:
    raise ValueError(f"[ERROR]-[dim_time][VERIFY]: expected 24 rows but {DIM_TIME_TABLE} has {written:,}")

print(f"[OK]-[dim_time][VERIFY]: {DIM_TIME_TABLE} has {written:,} rows")

#### `ref_community_area_boundaries`

A copy of `silver.community_area_boundaries` under its Gold name.

Steps:
1. ~~Enrich~~
2. Transform
3. Build
4. Load
5. Verify

In [0]:
# 2. [TRANSFORM] keep the five columns of the design

ref_community_area = spark.table(SILVER_CA_TABLE).select(
    "community_area_code",
    "community_area_name",
    "boundary_geometry",
    "shape_area_sqft",
    "shape_perimeter_ft"
)

ref_community_area.printSchema()

In [0]:
# 3. [BUILD] the check before the write: 77 rows, 77 distinct codes from 1 to 77, no NULL name

ref_rows, ref_codes, min_code, max_code, null_names = ref_community_area.agg(
    count("*"),
    count_distinct("community_area_code"),
    min("community_area_code"),
    max("community_area_code"),
    sum(col("community_area_name").isNull().cast("int")),
).collect()[0]

problems = []
if ref_rows != EXPECTED_COMMUNITY_AREAS: 
    problems.append(f"{ref_rows} rows (expected {EXPECTED_COMMUNITY_AREAS})")
if ref_codes != ref_rows: 
    problems.append(f"only {ref_codes} distinct community_area_code for {ref_rows} rows")
if (min_code, max_code) != (1, EXPECTED_COMMUNITY_AREAS): 
    problems.append(f"codes run from {min_code} to {max_code}")
if null_names: 
    problems.append(f"{null_names} NULL community_area_name")

if problems:
    raise ValueError(f"[ERROR]-[ref_community_area_boundaries][BUILD]: {'; '.join(problems)}")

print(f"[OK]-[ref_community_area_boundaries][BUILD]: {ref_rows} rows, codes 1..{max_code}, no NULL name")

In [0]:
# 4. [LOAD]

(
    ref_community_area.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(REF_COMMUNITY_AREA_TABLE)
)

In [0]:
# 5. [VERIFY] read the table back

written = spark.table(REF_COMMUNITY_AREA_TABLE).count()

if written != ref_rows:
    raise ValueError(f"[ERROR]-[ref_community_area_boundaries][VERIFY]: expected {ref_rows:,} rows but {REF_COMMUNITY_AREA_TABLE} has {written:,}")

print(f"[OK]-[ref_community_area_boundaries][VERIFY]: {REF_COMMUNITY_AREA_TABLE} has {written:,} rows")

### Non-Static Dimension

#### `dim_crime_type`

Steps:
1. ~~Enrich~~
2. Transform
3. Build
4. Load
5. Verify

In [0]:
# 2. [TRANSFORM]

natural = ["iucr", "primary_type", "description", "fbi_code"]

candidates = label_missing(df_raw.select(*natural)).distinct()

In [0]:
# 3. [BUILD] the combinations that are not in the table yet, with the next surrogate keys

new_rows, rows_before = build_new_dim_rows(DIM_CRIME_TYPE_TABLE, "crime_type_key", natural, candidates)

In [0]:
# 4. [LOAD]

new_rows.write.format("delta").mode("append").saveAsTable(DIM_CRIME_TYPE_TABLE)

In [0]:
verify_dim(DIM_CRIME_TYPE_TABLE, "crime_type_key", natural, candidates, rows_before)

#### `dim_location_description`

Steps:
1. ~~Enrich~~
2. Transform
3. Build
4. Load
5. Verify

In [0]:
# 2. [TRANSFORM]

natural = ["location_description"]

candidates = label_missing(df_raw.select(*natural)).distinct()

In [0]:
# 3. [BUILD] 

new_rows, rows_before = build_new_dim_rows(DIM_LOCATION_DESCRIPTION_TABLE, "location_description_key", natural, candidates)

In [0]:
# 4. [LOAD]

new_rows.write.format("delta").mode("append").saveAsTable(DIM_LOCATION_DESCRIPTION_TABLE)

In [0]:
# 5. [VERIFY]

verify_dim(DIM_LOCATION_DESCRIPTION_TABLE, "location_description_key", natural, candidates, rows_before)

#### `dim_block`

Steps: 
1. Enrich
2. Transform
3. Build
4. Load
5. Verify

In [0]:
# 1. [ENRICH] 
# (1) direction letter -> word (eg. N -> North)
# (2) street type abbreviation -> full name (AVE -> Avenue)

natural = ["block"]
attributes = ["block", "block_hundred", "direction", "street_name", "street_type"]

block_enriched = (
    df_raw.select(*attributes)
    .withColumn("direction", try_element_at(DIRECTION_MAP, col("direction")))
    .withColumn("street_type", initcap(try_element_at(STREET_TYPE_MAP, col("street_type"))))
)

enriched_rows = block_enriched.count()
if enriched_rows != raw_rows:
    raise ValueError(f"[ERROR]-[dim_block][ENRICH]: the batch has {raw_rows:,} rows but {enriched_rows:,} after the enrichment")

print(f"[OK]-[dim_block][ENRICH]: {enriched_rows:,} rows, none added or removed")

In [0]:
# 2. [TRANSFORM] NULL -> UNKNOWN, one row per block

candidates = label_missing(block_enriched).distinct()

In [0]:
# 3. [BUILD]

new_rows, rows_before = build_new_dim_rows(DIM_BLOCK_TABLE, "block_key", natural, candidates)

In [0]:
# 4. [LOAD]

new_rows.write.format("delta").mode("append").saveAsTable(DIM_BLOCK_TABLE)

In [0]:
# 5. [VERIFY]

verify_dim(DIM_BLOCK_TABLE, "block_key", natural, candidates, rows_before)

#### `dim_location`

Steps:
1. Enrich
2. Transform
3. Build
4. Load
5. Verify

In [0]:
# 1. [ENRICH] 
# (1) community_area_name from the reference table
# (2) district_name from the CPD list

if not spark.catalog.tableExists(REF_COMMUNITY_AREA_TABLE):
    raise ValueError(f"[ERROR]-[dim_location][ENRICH]: {REF_COMMUNITY_AREA_TABLE} does not exist - the static dimensions section did not run")

natural = ["beat", "district", "ward", "community_area"]

community_area_names = spark.table(REF_COMMUNITY_AREA_TABLE).select(col("community_area_code").alias("community_area"), col("community_area_name"))

location_enriched = (
    df_raw.select(*natural)
    .join(community_area_names, "community_area", "left")
    .withColumn("district_name", district_name_of("district"))
)

enriched_rows = location_enriched.count()
if enriched_rows != raw_rows:
    raise ValueError(f"[ERROR]-[dim_location][ENRICH]: the batch has {raw_rows:,} rows but {enriched_rows:,} after the enrichment ()")

print(f"[OK]-[dim_location][ENRICH]: {enriched_rows:,} rows, none added or removed")

In [0]:
# 2. [TRANSFORM] 

candidates = (
    label_missing(location_enriched)
    .select("beat", "district", "district_name", "ward", "community_area", "community_area_name")
    .distinct()
)

In [0]:
# 3. [BUILD]

new_rows, rows_before = build_new_dim_rows(DIM_LOCATION_TABLE, "location_key", natural, candidates)

In [0]:
# 4. [LOAD]

new_rows.write.format("delta").mode("append").saveAsTable(DIM_LOCATION_TABLE)

In [0]:
# 5. [VERIFY]

verify_dim(DIM_LOCATION_TABLE, "location_key", natural, candidates, rows_before)

#### `dim_arrest_status` (SCD Type 2)

Steps:
1. ~~Enrich~~
2. Transform
3. Build
4. Load
5. Verify

In [0]:
# 2. [TRANSFORM]

case_status = (
    df_raw.groupBy("case_number")
    .agg(
        expr("bool_or(arrest)").alias("arrest"),
        expr("bool_or(domestic)").alias("domestic"),
        max("updated_on").alias("effective_date")
    )
    .select(
        "case_number",
        when(col("arrest"), "Arrest Made").otherwise("No Arrest").alias("arrest_flag"),
        when(col("domestic"), "Domestic").otherwise("Not Domestic").alias("domestic_flag"),
        "effective_date"
    )
)

In [0]:
# 3. [BUILD] 

# (1) create the empty table on the first run

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DIM_ARREST_STATUS_TABLE} (
        arrest_status_key INT,
        case_number STRING,
        arrest_flag STRING,
        domestic_flag STRING,
        effective_date TIMESTAMP_NTZ,
        end_date TIMESTAMP_NTZ,
        is_current BOOLEAN,
        version_source STRING
    ) USING DELTA
""")

dim_is_empty = spark.table(DIM_ARREST_STATUS_TABLE).limit(1).count() == 0
first_version_source = "BACKFILL" if last_watermark is None else "NEW_CASE"

# (2) compare with the current row of each case
#   new_cases: no current row
#   changed: a current row exists and its flag differ.

current = (
    spark.table(DIM_ARREST_STATUS_TABLE).filter(col("is_current"))
    .select(
        "case_number",
        col("arrest_flag").alias("current_arrest_flag"),
        col("domestic_flag").alias("current_domestic_flag"),
        col("effective_date").alias("current_effective_date")
    )
)

compared = case_status.join(current, "case_number", "left") 

new_cases = compared.filter(col("current_arrest_flag").isNull())

changed = (
    compared
    .filter(
        col("current_arrest_flag").isNotNull() &
        ((col("arrest_flag") != col("current_arrest_flag")) | (col("domestic_flag") != col("current_domestic_flag")))
    )
    .withColumn("new_effective_date", greatest(col("current_effective_date"), col("effective_date")))
)

n_cases, n_new, n_changed = case_status.count(), new_cases.count(), changed.count()
print(f"[INFO]-[dim_arrest_status][BUILD]: {n_cases:,} cases in this batch: {n_new:,} new, {n_changed:,} with a changed status, {n_cases - n_new - n_changed:,} unchanged")

In [0]:
# 3. [BUILD] the rows to insert: a first version for every new case, and a new version for every changed case, with the next surrogate keys

to_insert = (
    new_cases.select("case_number", "arrest_flag", "domestic_flag", "effective_date", lit(first_version_source).alias("version_source"))
    .unionByName(
        changed.select("case_number", "arrest_flag", "domestic_flag", col("new_effective_date").alias("effective_date"), lit("DETECTED_CHANGE").alias("version_source"))
    )
)

max_key = spark.table(DIM_ARREST_STATUS_TABLE).agg(max("arrest_status_key")).collect()[0][0] or 0
to_insert = assign_keys(to_insert, "arrest_status_key", max_key, ["case_number", "version_source"])

n_to_insert = to_insert.count()

print(f"[OK]-[dim_arrest_status][BUILD]: {n_to_insert:,} rows to insert, {n_changed:,} current rows to close")

In [0]:
# 4. [LOAD]

if dim_is_empty:
    (
        to_insert.select("arrest_status_key", "case_number", "arrest_flag", "domestic_flag", "effective_date", lit(None).cast("timestamp_ntz").alias("end_date"), lit(True).alias("is_current"), "version_source")
        .write.format("delta").mode("append").saveAsTable(DIM_ARREST_STATUS_TABLE)
    )
    print(f"[INFO]-[dim_arrest_status][LOAD]: the dimension was empty - loaded {n_new} rows")
else:
    insert_rows = to_insert.select(
        lit(None).cast("string").alias("merge_key"),
        "arrest_status_key", "case_number", "arrest_flag", "domestic_flag", "effective_date", "version_source",
    )
    close_rows = changed.select(
        col("case_number").alias("merge_key"),
        lit(None).cast("int").alias("arrest_status_key"),
        "case_number",
        lit(None).cast("string").alias("arrest_flag"),
        lit(None).cast("string").alias("domestic_flag"),
        col("new_effective_date").alias("effective_date"),
        lit(None).cast("string").alias("version_source"),
    )
    insert_rows.unionByName(close_rows).createOrReplaceTempView("arrest_status_staged")

    spark.sql(f"""
        MERGE INTO {DIM_ARREST_STATUS_TABLE} AS t 
        USING arrest_status_staged AS s
        ON t.case_number = s.merge_key AND t.is_current = true         
        WHEN MATCHED THEN UPDATE SET end_date = s.effective_date, is_current = false
        WHEN NOT MATCHED AND s.arrest_status_key IS NOT NULL THEN INSERT 
            (arrest_status_key, case_number, arrest_flag, domestic_flag, effective_date, end_date, is_current, version_source)
        VALUES 
            (s.arrest_status_key, s.case_number, s.arrest_flag, s.domestic_flag, s.effective_date, NULL, true, s.version_source)
    """)

    print(f"[INFO]-[dim_arrest_status][LOAD]: merged {n_new:,} new cases and {n_changed:,} changed cases")

In [0]:
# 5. [VERIFY] dim_arrest_status

# (1) no case has more than one current row
# (2) every case of this batch has a current row, and its flags are the flags of this batch
# (3) a closed row has an end_date that is not before its effective_date; a current row has no end_date

dim = spark.table(DIM_ARREST_STATUS_TABLE)

two_current = dim.filter(col("is_current")).groupBy("case_number").count().filter(col("count") > 1).count()

current_now = dim.filter(col("is_current")).select("case_number", col("arrest_flag").alias("d_arrest"), col("domestic_flag").alias("d_domestic"))
checked = case_status.join(current_now, "case_number", "left")

no_current = checked.filter(col("d_arrest").isNull()).count()
wrong_status = checked.filter(col("d_arrest").isNotNull() & ((col("arrest_flag") != col("d_arrest")) | (col("domestic_flag") != col("d_domestic")))).count()

bad_history = dim.filter(
    (~col("is_current") & (col("end_date").isNull() | (col("end_date") < col("effective_date"))))
    | (col("is_current") & col("end_date").isNotNull())
).count()

problems = {"cases with two current rows": two_current, "cases of this batch without a current row": no_current,
            "cases whose current row differs from this batch": wrong_status, "rows with a wrong end_date / is_current": bad_history}
problems = {k: v for k, v in problems.items() if v > 0}

if problems:
    raise ValueError(f"[ERROR]-[dim_arrest_status][VERIFY]: {problems}")

print(f"[OK]-[dim_arrest_status][VERIFY]: {dim.count():,} rows, one current row per case, history is consistent")

## Fact

### `fact_crime`

Steps:
1. ~~Enrich~~
2. Transform
3. Build
4. Load
5. Verify

In [0]:
# 2. [TRANSFORM]

batch = (
    label_missing(df_raw)
    .withColumn("incident_date_key", date_format("incident_date", "yyyyMMdd").cast("int"))
    .withColumn("incident_time_key", col("incident_hour").cast("tinyint"))
    .withColumn("crime_count", lit(1).cast("tinyint"))
)

In [0]:
# 3. [BUILD] look up the FK. A dimension is joined on its natural key 

def keys_of(table: str, natural_columns: list, key_column: str):
    return spark.table(table).select(*natural_columns, key_column)

arrest_status_now = (
    spark.table(DIM_ARREST_STATUS_TABLE)
    .filter(col("is_current"))
    .select("case_number", "arrest_status_key")
)

location_natural = ["beat", "district", "ward", "community_area"]
crime_type_natural = ["iucr", "primary_type", "description", "fbi_code"]

df_fact = (
    batch
    .join(keys_of(DIM_LOCATION_TABLE, location_natural, "location_key"), location_natural, "left")
    .join(keys_of(DIM_BLOCK_TABLE, ["block"], "block_key"), ["block"], "left")
    .join(keys_of(DIM_CRIME_TYPE_TABLE, crime_type_natural, "crime_type_key"), crime_type_natural, "left")
    .join(keys_of(DIM_LOCATION_DESCRIPTION_TABLE, ["location_description"], "location_description_key"), ["location_description"], "left")
    .join(arrest_status_now, "case_number", "left")
    .select(
        "id", "case_number",
        "incident_date_key", "incident_time_key",
        "location_key", "block_key", "crime_type_key", "arrest_status_key", "location_description_key",
        "x_coordinate", "y_coordinate", "latitude", "longitude",
        "crime_count", "updated_on",
    )
)

In [0]:
# 4. [LOAD] first run: the table does not exist yet, so it is created from this batch.
# Later: MERGE on id

if not spark.catalog.tableExists(FACT_TABLE):
    df_fact.write.format("delta").saveAsTable(FACT_TABLE)
    inserted, updated = raw_rows, 0
    print(f"[INFO]-[fact_crime][LOAD]: {FACT_TABLE} did not exist - created it with {raw_rows:,} rows")
else: 
    df_fact.createOrReplaceTempView("fact_batch")

    merge_sql = f"""
        MERGE INTO {FACT_TABLE} AS t
        USING fact_batch AS s
        ON t.id = s.id
        WHEN MATCHED AND s.updated_on >= t.updated_on THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """

    result = spark.sql(merge_sql).collect()[0]
    inserted, updated = result["num_inserted_rows"], result["num_updated_rows"]

print(f"[INFO]-[fact_crime][LOAD]: {inserted:,} inserted, {updated:,} updated")

In [0]:
# 5. [VERIFY] one row per id in the whole table, and every id of this batch is in it

fact = spark.table(FACT_TABLE)

total_rows, total_ids = fact.agg(count("*"), count_distinct("id")).collect()[0]

if total_rows != total_ids:
    raise ValueError(f"[ERROR]-[fact_crime][VERIFY]: {total_rows:,} rows but {total_ids:,} distinct id")

missing_after_write = df_raw.select("id").join(fact.select("id"), "id", "left_anti").count()

if missing_after_write > 0:
    raise ValueError(f"[ERROR]-[fact_crime][VERIFY]: {missing_after_write:,} ids of this batch are not in {FACT_TABLE}")

print(f"[OK]-[fact_crime][VERIFY]: {total_rows:,} rows in {FACT_TABLE}, one per id")

# Move the Watermark

In [0]:
# create the control table on the first run

if not spark.catalog.tableExists(CONTROL_TABLE):
    spark.createDataFrame([], CONTROL_SCHEMA).write.format("delta").saveAsTable(CONTROL_TABLE)
    print(f"[INFO]-[WATERMARK]: {CONTROL_TABLE} did not exist - created it")

In [0]:
# one row: insert it, or move its watermark forward

(
    spark.createDataFrame([("crimes", batch_end)], "table_name string, last_ingested_at timestamp")
    .withColumn("updated_at", current_timestamp())
    .createOrReplaceTempView("control_batch")
)

control_sql = f"""
    MERGE INTO {CONTROL_TABLE} AS t
    USING control_batch AS s
    ON t.table_name = s.table_name
    WHEN MATCHED AND s.last_ingested_at >= t.last_ingested_at THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
"""

spark.sql(control_sql)

In [0]:
# read the watermark back: it must be the end of this batch (FAIL otherwise)

saved_watermark = get_last_watermark()

if saved_watermark != batch_end:
    raise ValueError(f"[ERROR]-[WATERMARK]: expected {batch_end} in {CONTROL_TABLE}, found {saved_watermark}")

print(f"[OK]-[WATERMARK][VERIFY]: {CONTROL_TABLE} says crimes is loaded up to {saved_watermark}")